In [ ]:
import pandas as pd
import numpy as np
import requests

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 38.03,
    "longitude": -78.48,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "weather_code"
    ],
    "temperature_unit": "fahrenheit",
    "timezone": "America/New_York",
    "past_days": 7,
    "forecast_days": 3
}

response = requests.get(url, params=params)

data = response.json()

In [ ]:
weather = pd.DataFrame(data["hourly"])

weather.head()

In [ ]:
weather.dtypes

In [ ]:
weather = weather.rename(columns={
    "time": "datetime",
    "temperature_2m": "temperature_f",
    "relative_humidity_2m": "humidity_pct",
    "precipitation": "precipitation_mm",
    "wind_speed_10m": "wind_speed_kmh",
    "weather_code": "weather_code"
})

In [ ]:
weather

In [ ]:
weather.describe()

In [ ]:
weather["rain"] = np.where(
    weather["precipitation_mm"] > 0,
    "Yes",
    "No"
)

In [ ]:
weather["rain"].value_counts(
    normalize=True
)

In [ ]:
import plotly.express as px

In [ ]:
fig = px.line(
    weather,
    x="datetime",
    y="temperature_f",
    title="Temperature Over Time",
    labels={
        "datetime": "Date and Time",
        "temperature_f": "Temperature (°F)"
    },
    hover_data=[
        "humidity_pct",
        "precipitation_mm",
        "wind_speed_kmh",
        "weather_code"
    ]
)

fig.show()

In [ ]:
fig = px.scatter(
    weather,
    x="humidity_pct",
    y="temperature_f",
    color="rain",
    hover_data=[
        "datetime",
        "precipitation_mm",
        "wind_speed_kmh"
    ],
    title="Temperature vs. Humidity: Rain vs. No Rain",
    labels={
        "humidity_pct": "Relative Humidity (%)",
        "temperature_f": "Temperature (°F)"
    }
)

fig.show()

In [ ]:
locations = pd.DataFrame({
    "city": [
        "Charlottesville",
        "Crozet",
        "Earlysville",
        "Scottsville",
        "Palmyra",
        "Waynesboro"
    ],
    "latitude": [
        38.0293,
        38.0667,
        38.1575,
        37.7996,
        37.8610,
        38.0687
    ],
    "longitude": [
        -78.4767,
        -78.6996,
        -78.4828,
        -78.4956,
        -78.2637,
        -78.8894
    ]
})


In [ ]:
weather_list = []

for _, row in locations.iterrows():

    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "weather_code"
        ],
        "temperature_unit": "fahrenheit",
        "timezone": "America/New_York",
        "past_days": 7,
        "forecast_days": 3
    }

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params=params
    )

    data = response.json()

    temp = pd.DataFrame(data["hourly"])

    temp["city"] = row["city"]
    temp["latitude"] = row["latitude"]
    temp["longitude"] = row["longitude"]

    weather_list.append(temp)


weather = pd.concat(
    weather_list,
    ignore_index=True
)

In [ ]:
weather = weather.rename(columns={
    "time": "datetime",
    "temperature_2m": "temperature_f",
    "relative_humidity_2m": "humidity_pct",
    "precipitation": "precipitation_mm",
    "wind_speed_10m": "wind_speed_kmh",
    "weather_code": "weather_code"
})

In [ ]:
weather["datetime"] = pd.to_datetime(
    weather["datetime"]
)

In [ ]:
weather.head()

In [ ]:
latest_weather = (
    weather
    .sort_values("datetime")
    .groupby("city")
    .tail(1)
)

In [ ]:
latest_weather[
    ["city", "temperature_f", "humidity_pct"]
]

In [ ]:
fig = px.scatter_map(
    latest_weather,
    lat="latitude",
    lon="longitude",
    color="temperature_f",
    size="temperature_f",
    hover_name="city",
    hover_data={
        "temperature_f": True,
        "humidity_pct": True,
        "precipitation_mm": True,
        "wind_speed_kmh": True,
        "latitude": False,
        "longitude": False
    },
    zoom=8,
    title="Temperature Across Central Virginia",
    labels={
        "temperature_f": "Temperature (°F)"
    }
)

fig.update_layout(
    map_style="open-street-map"
)

fig.show()